# English-Luganda Translation - SIMPLE & FAST

**One dataset, minimal code, runs in 10 minutes on T4**

Key differences from complex version:
- Uses MarianMT (battle-tested, smaller)
- Single CSV input (no combining 5 files)
- BATCH_SIZE=2 (guaranteed to fit T4)
- 2 epochs instead of 3
- No quantization tricks or memory optimization layers
- Simple error handling
- ~5 minutes training time

In [ ]:
# Cell 0: Install & Setup
!pip install --upgrade pip -q
!pip install transformers datasets torch pandas scikit-learn sacrebleu tqdm -q

import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments
from datasets import Dataset
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import os
import json
from datetime import datetime

print("Setup complete")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")

In [ ]:
# Cell 1: Load CSV Data
# Upload ONE CSV file to /content/ with columns: 'english' and 'luganda'
# Example filename: kambale_train.csv

import os

print("Available CSV files in /content/:")
csv_files = [f for f in os.listdir('/content/') if f.endswith('.csv')]
for f in csv_files:
    size = os.path.getsize(f'/content/{f}') / (1024*1024)
    print(f"  - {f} ({size:.1f} MB)")

# Load data
csv_file = csv_files[0] if csv_files else None

if csv_file:
    df = pd.read_csv(f'/content/{csv_file}')
    print(f"\nLoaded: {csv_file}")
    print(f"Rows: {len(df)}")
    print(f"Columns: {df.columns.tolist()}")
else:
    print("No CSV found. Creating sample data...")
    df = pd.DataFrame({
        'english': ['Hello', 'Thank you', 'Good morning', 'How are you', 'I love Uganda'] * 4,
        'luganda': ['Agandi', 'Webale', 'Ku makya', 'Oli otya', 'Njagira Uganda'] * 4
    })
    print(f"Created sample data: {len(df)} rows")

# Check columns
if 'english' not in df.columns or 'luganda' not in df.columns:
    print(f"ERROR: Expected columns 'english' and 'luganda', got {df.columns.tolist()}")
else:
    print(f"\nData preview:")
    print(df.head(3))

dataset_df = df.copy()

In [ ]:
# Cell 2: Clean Data
import re

def normalize_text(text):
    text = str(text).strip().lower()
    text = ' '.join(text.split())
    return text

# Clean
df = dataset_df.copy()
print(f"Starting: {len(df)} rows")

df = df.dropna()
print(f"After removing NaN: {len(df)} rows")

df['english'] = df['english'].apply(normalize_text)
df['luganda'] = df['luganda'].apply(normalize_text)

df = df[(df['english'].str.len() > 0) & (df['luganda'].str.len() > 0)]
print(f"After removing empty: {len(df)} rows")

df = df.drop_duplicates(subset=['english', 'luganda'])
print(f"After removing duplicates: {len(df)} rows")

dataset_clean = df.reset_index(drop=True)
print(f"\nCleaned data:")
print(dataset_clean.head(3))

In [ ]:
# Cell 3: Load Model (MarianMT)
# MarianMT is smaller, faster, and more reliable than NLLB for this task

MODEL_NAME = "Helsinki-NLP/opus-mt-en-mul"  # 200M, optimized for English->Multilingual

print(f"Loading {MODEL_NAME}...")
print("(This downloads ~400MB, takes 1-2 minutes first time)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"Model loaded successfully")
print(f"Device: {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Cell 4: Prepare Data

print(f"Dataset size: {len(dataset_clean)}")

# Split
if len(dataset_clean) < 10:
    train_data = dataset_clean.iloc[:-2]
    val_data = dataset_clean.iloc[-2:-1]
    test_data = dataset_clean.iloc[-1:]
else:
    train_data, temp = train_test_split(dataset_clean, test_size=0.2, random_state=42)
    val_data, test_data = train_test_split(temp, test_size=0.5, random_state=42)

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

# Create HF Datasets
def make_dataset(df):
    return Dataset.from_dict({
        'english': df['english'].tolist(),
        'luganda': df['luganda'].tolist()
    })

train_dataset = make_dataset(train_data)
val_dataset = make_dataset(val_data)
test_dataset = make_dataset(test_data)

# Tokenize
def preprocess(examples):
    inputs = tokenizer([f"en->lg: {t}" for t in examples['english']], 
                       max_length=128, truncation=True, padding='max_length')
    labels = tokenizer(examples['luganda'], 
                      max_length=100, truncation=True, padding='max_length')
    inputs['labels'] = labels['input_ids']
    return inputs

train_tokenized = train_dataset.map(preprocess, batched=True, remove_columns=['english', 'luganda'])
val_tokenized = val_dataset.map(preprocess, batched=True, remove_columns=['english', 'luganda'])
test_tokenized = test_dataset.map(preprocess, batched=True, remove_columns=['english', 'luganda'])

print(f"Tokenization complete")

In [ ]:
# Cell 5: Train

output_dir = "/content/model_output"
os.makedirs(output_dir, exist_ok=True)

args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=2,
    per_device_train_batch_size=2,  # Small = guaranteed to fit T4
    per_device_eval_batch_size=2,
    learning_rate=2e-5,
    warmup_steps=50,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    seed=42
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
)

print("Starting training...")
start = datetime.now()

try:
    trainer.train()
    elapsed = (datetime.now() - start).total_seconds() / 60
    print(f"\nTraining complete! ({elapsed:.1f} minutes)")
except Exception as e:
    print(f"Error during training: {e}")
    print(f"If OOM: data is too large or batch size needs reduction")

In [ ]:
# Cell 6: Evaluate & Save

print("Generating predictions on test set...")
model.eval()

predictions = []
references = test_data['luganda'].tolist()

with torch.no_grad():
    for i in tqdm(range(len(test_data))):
        text = test_data['english'].iloc[i]
        inputs = tokenizer(f"en->lg: {text}", return_tensors="pt", max_length=128, truncation=True).to(device)
        
        output = model.generate(**inputs, max_length=100, num_beams=2)
        pred = tokenizer.decode(output[0], skip_special_tokens=True)
        predictions.append(pred)

# BLEU Score
try:
    from sacrebleu import corpus_bleu
    bleu = corpus_bleu(predictions, [references])
    print(f"\nBLEU Score: {bleu.score:.2f}")
except Exception as e:
    print(f"BLEU calculation error: {e}")

# Show examples
print(f"\nSample predictions:")
for i in range(min(3, len(test_data))):
    print(f"\n  EN: {test_data['english'].iloc[i]}")
    print(f"  REF: {test_data['luganda'].iloc[i]}")
    print(f"  PRED: {predictions[i]}")

# Save
print(f"\nSaving model...")
save_dir = "/content/final_model"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"Model saved to {save_dir}")
print(f"\nDone! Download from Files panel.")